# PySpark drills 5: Spark SQL and the catalog

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-05-spark-sql-and-the-catalog.ipynb)

Practice for [PySpark 5: Spark SQL and the catalog](pyspark-05-spark-sql-and-the-catalog.ipynb). Answer each exercise in Spark SQL where you can. The `customers` and `orders` tables are created above.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Exercise 1: Revenue per status with HAVING

**Difficulty:** Easy · about 5 min

Register `orders` as a temporary view and, in SQL, return `status` and `revenue` (sum of `amount`) for statuses
whose revenue is over 200, ordered by `status`.

**Expected output**

| status | revenue |
| --- | --- |
| paid | 764.00 |
| placed | 265.25 |
| shipped | 1509.90 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
orders.createOrReplaceTempView("orders")
result = spark.sql("""
    SELECT status, sum(amount) AS revenue FROM orders
    GROUP BY status HAVING sum(amount) > 200 ORDER BY status
""")
result.show()

## Exercise 2: A parameterized query

**Difficulty:** Medium · about 10 min

Write `orders_with_status(status)`, which runs one SQL query with a named parameter (no string formatting of the
value) and returns `order_id` and `amount` for that status, ordered by `order_id`. Call it for `shipped`.

**Expected output**

| order_id | amount |
| --- | --- |
| 2 | 84.50 |
| 6 | 149.00 |
| 9 | 1180.00 |
| 12 | 96.40 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
orders.createOrReplaceTempView("orders")


def orders_with_status(status):
    return spark.sql(
        "SELECT order_id, amount FROM orders WHERE status = :status ORDER BY order_id", args={"status": status}
    )


result = orders_with_status("shipped")
result.show()

## Exercise 3: Each customer's second-largest order

**Difficulty:** Medium · about 15 min

With a CTE and a window function in SQL, return `customer_id`, `order_id` and `amount` of each customer's
second-largest order (ties broken by the smaller `order_id`). Customers with one order are left out. Order by
`customer_id`.

**Expected output**

| customer_id | order_id | amount |
| --- | --- | --- |
| 1 | 2 | 84.50 |
| 2 | 5 | 55.25 |
| 3 | 6 | 149.00 |
| 4 | 8 | 89.90 |
| 6 | 12 | 96.40 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
orders.createOrReplaceTempView("orders")
result = spark.sql("""
    WITH ranked AS (
        SELECT customer_id, order_id, amount,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
        FROM orders
    )
    SELECT customer_id, order_id, amount FROM ranked WHERE rn = 2 ORDER BY customer_id
""")
result.show()

## Exercise 4: Overwrite one partition of a table

**Difficulty:** Medium · about 15 min

Create a Parquet table `drill.daily_orders (orders BIGINT, order_date DATE)` partitioned by `order_date`. Insert the
order count of every date, then replace 12 January with a count of 0 using `INSERT OVERWRITE ... PARTITION`. Return
the table ordered by `order_date`.

**Expected output**

| order_date | orders |
| --- | --- |
| 2026-01-05 | 3 |
| 2026-01-06 | 1 |
| 2026-01-07 | 2 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 0 |
| 2026-01-13 | 1 |
| 2026-01-14 | 1 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
orders.createOrReplaceTempView("orders")
spark.sql("CREATE DATABASE IF NOT EXISTS drill")
spark.sql("DROP TABLE IF EXISTS drill.daily_orders")
spark.sql("CREATE TABLE drill.daily_orders (orders BIGINT, order_date DATE) USING parquet PARTITIONED BY (order_date)")
spark.sql("INSERT INTO drill.daily_orders SELECT count(*), to_date(order_ts) FROM orders GROUP BY to_date(order_ts)")
spark.sql("INSERT OVERWRITE drill.daily_orders PARTITION (order_date = DATE'2026-01-12') SELECT 0")
result = spark.table("drill.daily_orders").select("order_date", "orders").orderBy("order_date")
result.show()

## Exercise 5: Managed or external?

**Difficulty:** Easy · about 10 min

Save `customers` as a managed table `drill.customers_m` and as an external table `drill.customers_x` stored under
`WORK_DIR`. Return a DataFrame with `table` and `type` (as reported by `DESCRIBE EXTENDED`), ordered by `table`.

**Expected output**

| table | type |
| --- | --- |
| customers_m | MANAGED |
| customers_x | EXTERNAL |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS drill")
customers.write.mode("overwrite").saveAsTable("drill.customers_m")
customers.write.mode("overwrite").option("path", os.path.join(WORK_DIR, "drill_customers_x")).saveAsTable("drill.customers_x")
rows = [
    (name, spark.sql(f"DESCRIBE EXTENDED drill.{name}").where("col_name = 'Type'").first().data_type)
    for name in ["customers_m", "customers_x"]
]
result = spark.createDataFrame(rows, "table string, type string").orderBy("table")
result.show()

**Next:** [PySpark 6: joins](pyspark-06-joins.ipynb)